# Phase 1 - Kiểm chứng toán học DTQRW

Notebook kiểm tra symbolic unitarity, spectrum của evolution operator, phân phối Hadamard walk, ballistic variance và classical baseline. Quy ước: coin `up` dịch phải, coin `down` dịch trái, initial coin $(|\uparrow\rangle+i|\downarrow\rangle)/\sqrt{2}$.

In [1]:
from pathlib import Path
from math import comb
import json

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

REPORTS = Path('reports')
REPORTS.mkdir(parents=True, exist_ok=True)
np.set_printoptions(precision=6, suppress=True)
print('NumPy:', np.__version__)
print('SymPy:', sp.__version__)

NumPy: 2.4.6
SymPy: 1.14.0


## 1. Unitarity và eigenvalues

Trên đường thẳng vô hạn, shift là một hoán vị của basis. Để tạo ma trận hữu hạn, ta dùng cycle có 11 vị trí; periodic shift vẫn unitary. Khi $U=S(H\otimes I)$, cần có $U^\dagger U=I$ và mọi eigenvalue nằm trên unit circle.

In [2]:
sqrt2 = sp.sqrt(2)
H_symbolic = sp.Matrix([[1, 1], [1, -1]]) / sqrt2
symbolic_identity = sp.simplify(H_symbolic.H * H_symbolic)
assert symbolic_identity == sp.eye(2)
print('H^dagger H =')
sp.pprint(symbolic_identity)

H = np.array([[1.0, 1.0], [1.0, -1.0]], dtype=complex) / np.sqrt(2.0)
n_positions = 11
coin_full = np.kron(H, np.eye(n_positions, dtype=complex))
shift = np.zeros((2 * n_positions, 2 * n_positions), dtype=complex)
idx = lambda coin, x: coin * n_positions + x
for x in range(n_positions):
    shift[idx(0, (x + 1) % n_positions), idx(0, x)] = 1.0
    shift[idx(1, (x - 1) % n_positions), idx(1, x)] = 1.0

U = shift @ coin_full
identity = np.eye(2 * n_positions, dtype=complex)
unitarity_norm = float(np.linalg.norm(U.conj().T @ U - identity, ord='fro'))
eigenvalues = np.linalg.eigvals(U)
eigenvalue_radius_error = float(np.max(np.abs(np.abs(eigenvalues) - 1.0)))
print(f'||U^dagger U - I||_F = {unitarity_norm:.3e}')
print(f'max ||lambda|-1|       = {eigenvalue_radius_error:.3e}')
assert unitarity_norm < 1e-10
assert eigenvalue_radius_error < 1e-10

H^dagger H =
[1  0]
[    ]
[0  1]
||U^dagger U - I||_F = 1.047e-15
max ||lambda|-1|       = 1.443e-15


## 2. Exact statevector và CRW baseline

Lattice có $2T+1$ sites nên chứa toàn bộ light cone sau $T$ bước, không cần wrap-around. QRW được tiến hóa bằng amplitude; CRW dùng phân phối binomial chính xác.

In [3]:
def qrw_distribution(steps):
    positions = np.arange(-steps, steps + 1)
    psi = np.zeros((2, positions.size), dtype=complex)
    psi[:, steps] = np.array([1.0, 1.0j]) / np.sqrt(2.0)
    variances = []
    norm_errors = []
    symmetry_errors = []

    for _ in range(steps):
        after_coin = H @ psi
        shifted = np.zeros_like(psi)
        shifted[0, 1:] = after_coin[0, :-1]
        shifted[1, :-1] = after_coin[1, 1:]
        psi = shifted
        probability = np.sum(np.abs(psi) ** 2, axis=0)
        mean = float(probability @ positions)
        variances.append(float(probability @ (positions - mean) ** 2))
        norm_errors.append(abs(float(probability.sum()) - 1.0))
        symmetry_errors.append(float(np.max(np.abs(probability - probability[::-1]))))

    return positions, probability, np.array(variances), norm_errors, symmetry_errors


def crw_distribution(steps):
    positions = np.arange(-steps, steps + 1)
    probability = np.zeros(positions.size, dtype=float)
    for i, x in enumerate(positions):
        if (steps + x) % 2 == 0:
            probability[i] = comb(steps, (steps + x) // 2) / (2 ** steps)
    return positions, probability


def distribution_variance(positions, probability):
    mean = float(probability @ positions)
    return float(probability @ (positions - mean) ** 2)

print('Simulation helpers loaded.')

Simulation helpers loaded.


## 3. Variance scaling và hiệu chỉnh công thức

Với Hadamard walk đối xứng chuẩn, hệ số đúng là $1-1/\sqrt{2}\approx0.292893$, không phải $1/2$. Vì vậy $\operatorname{Var}(X_T)=(1-1/\sqrt{2})T^2+O(T)$. Cell sau so sánh exact statevector, asymptotic formula và CRW.

In [4]:
T = 100
positions, qrw_probability, qrw_variances, norm_errors, symmetry_errors = qrw_distribution(T)
_, crw_probability = crw_distribution(T)
qrw_variance = distribution_variance(positions, qrw_probability)
crw_variance = distribution_variance(positions, crw_probability)
variance_ratio = qrw_variance / crw_variance
asymptotic_coefficient = 1.0 - 1.0 / np.sqrt(2.0)
corrected_asymptotic_variance = asymptotic_coefficient * T ** 2
plan_candidate_variance = T ** 2 / 2.0 - T / 4.0

fit_steps = np.arange(20, T + 1)
qrw_scaling_exponent = float(np.polyfit(np.log(fit_steps), np.log(qrw_variances[19:]), 1)[0])
crw_scaling_exponent = 1.0

print(f'T = {T}')
print(f'Exact QRW variance                = {qrw_variance:.6f}')
print(f'Corrected asymptotic variance     = {corrected_asymptotic_variance:.6f}')
print(f'Original-plan candidate variance = {plan_candidate_variance:.6f}')
print(f'Exact CRW variance                = {crw_variance:.6f}')
print(f'QRW / CRW variance ratio          = {variance_ratio:.6f}')
print(f'Fitted QRW log-log exponent       = {qrw_scaling_exponent:.6f}')
print(f'Max normalization error           = {max(norm_errors):.3e}')
print(f'Max symmetry error                = {max(symmetry_errors):.3e}')

assert max(norm_errors) < 1e-12
assert max(symmetry_errors) < 1e-12
assert variance_ratio > 1.5
assert 1.8 < qrw_scaling_exponent < 2.1

T = 100
Exact QRW variance                = 2929.422331
Corrected asymptotic variance     = 2928.932188
Original-plan candidate variance = 4975.000000
Exact CRW variance                = 100.000000
QRW / CRW variance ratio          = 29.294223
Fitted QRW log-log exponent       = 1.997996
Max normalization error           = 1.754e-14
Max symmetry error                = 1.527e-16


## 4. Monte Carlo endpoint measurement

Statevector cho phân phối chính xác. Monte Carlo ở đây mô phỏng 1000 phép đo độc lập từ endpoint distribution, không biến evolution unitary thành classical trajectories.

In [5]:
rng = np.random.default_rng(2026)
n_measurements = 1000
qrw_samples = rng.choice(positions, size=n_measurements, p=qrw_probability)
crw_samples = rng.choice(positions, size=n_measurements, p=crw_probability)
qrw_mc_variance = float(np.var(qrw_samples))
crw_mc_variance = float(np.var(crw_samples))
print(f'QRW sample variance ({n_measurements} measurements) = {qrw_mc_variance:.6f}')
print(f'CRW sample variance ({n_measurements} measurements) = {crw_mc_variance:.6f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(positions, qrw_probability, label='QRW Hadamard', linewidth=1.4)
axes[0].plot(positions, crw_probability, label='CRW', linewidth=1.4)
axes[0].set(title='Endpoint distributions at T=100', xlabel='Position', ylabel='Probability')
axes[0].legend()
axes[0].grid(alpha=0.25)

all_steps = np.arange(1, T + 1)
axes[1].loglog(all_steps, qrw_variances, label='QRW exact variance')
axes[1].loglog(all_steps, all_steps, label='CRW variance = T')
axes[1].loglog(all_steps, asymptotic_coefficient * all_steps ** 2, '--', label='(1-1/sqrt(2)) T^2')
axes[1].set(title='Variance scaling', xlabel='T', ylabel='Variance')
axes[1].legend()
axes[1].grid(alpha=0.25, which='both')
fig.tight_layout()
figure_path = REPORTS / 'theory_verification.png'
fig.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.close(fig)
print('Saved:', figure_path)

QRW sample variance (1000 measurements) = 2873.058396
CRW sample variance (1000 measurements) = 99.901104
Saved: reports\theory_verification.png


In [6]:
results = {
    'unitarity_frobenius_norm': unitarity_norm,
    'max_eigenvalue_radius_error': eigenvalue_radius_error,
    'max_probability_normalization_error': float(max(norm_errors)),
    'max_symmetry_error': float(max(symmetry_errors)),
    'T': T,
    'qrw_variance_exact': qrw_variance,
    'qrw_variance_asymptotic_corrected': corrected_asymptotic_variance,
    'original_plan_candidate_variance': plan_candidate_variance,
    'crw_variance_exact': crw_variance,
    'variance_ratio_qrw_to_crw': variance_ratio,
    'qrw_scaling_exponent_fit_T20_100': qrw_scaling_exponent,
    'qrw_monte_carlo_variance_n1000': qrw_mc_variance,
    'crw_monte_carlo_variance_n1000': crw_mc_variance,
    'random_seed': 2026,
}
results_path = REPORTS / 'theory_verification_results.json'
results_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(results, indent=2))
print('\nALL PHASE 1 MATHEMATICAL CHECKS PASSED')

{
  "unitarity_frobenius_norm": 1.0467540502466446e-15,
  "max_eigenvalue_radius_error": 1.4432899320127035e-15,
  "max_probability_normalization_error": 1.7541523789077473e-14,
  "max_symmetry_error": 1.5265566588595902e-16,
  "T": 100,
  "qrw_variance_exact": 2929.4223307939237,
  "qrw_variance_asymptotic_corrected": 2928.9321881345254,
  "original_plan_candidate_variance": 4975.0,
  "crw_variance_exact": 100.0,
  "variance_ratio_qrw_to_crw": 29.294223307939237,
  "qrw_scaling_exponent_fit_T20_100": 1.9979960820955938,
  "qrw_monte_carlo_variance_n1000": 2873.0583960000004,
  "crw_monte_carlo_variance_n1000": 99.901104,
  "random_seed": 2026
}

ALL PHASE 1 MATHEMATICAL CHECKS PASSED


## 5. Diễn giải

Các kiểm tra xác nhận evolution operator unitary, spectrum nằm trên unit circle, phân phối được chuẩn hóa và đối xứng, QRW tăng trưởng ballistic trong khi CRW tăng trưởng diffusive. Công thức variance trong kế hoạch ban đầu đã được thay bằng hệ số Hadamard đúng. Kết quả này là kiểm tra implementation, chưa phải bằng chứng QRW phù hợp dữ liệu thị trường.